# 2. Manipulation de données avec Polars

Dans ce notebook, on remplace `danfojs-node` par `nodejs-polars`, une bibliothèque de DataFrames rapide et moderne.

## 2.1 Charger un fichier CSV

In [ ]:
import pl from "nodejs-polars";

const df = pl.readCSV("../data/titanic.csv");

// Afficher les 5 premières lignes
console.log(df.head(5).toString());

// Dimensions
console.log("Dimensions :", df.shape);

## 2.2 Informations sur les colonnes

In [ ]:
// Liste des colonnes
console.log("Colonnes :", df.columns);

// Types des colonnes
console.log("Types :");
console.log(df.schema);

## 2.3 Sélectionner des colonnes

In [ ]:
// Sélectionner quelques colonnes
const selection = df.select("Name", "Sex", "Age", "Survived");
console.log(selection.head(5).toString());

## 2.4 Filtrer les lignes

In [ ]:
// Passagers de première classe
const firstClass = df.filter(pl.col("Pclass").eq(1));
console.log(firstClass.head(5).toString());

// Passagers ayant survécu et âgés de plus de 50 ans
const survivedOld = df.filter(
  pl.col("Survived").eq(1).and(pl.col("Age").gt(50))
);
console.log(survivedOld.head(5).toString());

## 2.5 Trier les lignes

In [ ]:
const sorted = df.sort("Fare", true); // true = décroissant
console.log(sorted.head(10).select("Name", "Fare", "Pclass").toString());

## 2.6 Agréger des données

In [ ]:
// Taux de survie par classe
const survivalByClass = df
  .groupBy("Pclass")
  .agg(pl.col("Survived").mean().alias("survival_rate"));

console.log(survivalByClass.sort("Pclass").toString());

// Nombre de passagers et âge moyen par sexe
const statsBySex = df
  .groupBy("Sex")
  .agg(
    pl.col("PassengerId").count().alias("count"),
    pl.col("Age").mean().alias("mean_age"),
    pl.col("Fare").mean().alias("mean_fare"),
  );

console.log(statsBySex.toString());

## 2.7 Créer une nouvelle colonne

In [ ]:
// Colonne indiquant si le passager est un enfant (< 18 ans)
const dfWithChild = df.withColumns(
  pl.col("Age").lt(18).alias("is_child")
);

console.log(dfWithChild.head(5).select("Name", "Age", "is_child").toString());

// Taux de survie des enfants vs adultes
const survivalByChild = dfWithChild
  .groupBy("is_child")
  .agg(pl.col("Survived").mean().alias("survival_rate"));

console.log(survivalByChild.toString());

## 2.8 Gérer les valeurs manquantes

In [ ]:
// Nombre de valeurs manquantes par colonne
const nullCounts = df.nullCount();
console.log(nullCounts.toString());

// Supprimer les lignes avec un âge manquant
const dfNoMissingAge = df.dropNulls("Age");
console.log("Avant :", df.shape, "Après :", dfNoMissingAge.shape);

## 2.9 Exporter vers CSV

In [ ]:
// Sauvegarder le résultat d'une analyse
survivalByClass.writeCSV("../data/titanic_survival_by_class.csv");
console.log("Fichier exporté.");